In [ ]:
!pip install nevergrad

import matplotlib.pyplot as plt
import nevergrad
import numpy
import pandas
import scipy.stats
import seaborn
import sklearn
import sklearn.base
import sklearn.inspection
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline
import sklearn.preprocessing
from IPython.core.display import display

%matplotlib inline

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-ames.git

# Linear regression

## Introduction

Linear regression is interesting to code from scratch: it's not too hard but it's a good introduction to many machine learning concepts.

Let's load the data.

In [ ]:
train_df = pandas.read_csv("dataset-ames/train.csv", index_col="Id")

In [ ]:
display(train_df)

## Used feature

For now, we'll only use one feature: `GrLivArea`. `SalePrice` will be our target.

In [ ]:
seaborn.jointplot(x="GrLivArea", y="SalePrice", data=train_df)
plt.show()

In [ ]:
# Standardize x
x_scaler = sklearn.preprocessing.StandardScaler()
x = x_scaler.fit_transform(train_df[["GrLivArea"]])[:, 0]

# Standardize y
y_scaler = sklearn.preprocessing.StandardScaler()
y = y_scaler.fit_transform(train_df[["SalePrice"]])[:, 0]

## Linear regression example

Linear regression means finding the best line to fit our data. Here we could for example find:

In [ ]:
seaborn.regplot(x=x, y=y, scatter_kws=dict(alpha=0.10), line_kws=dict(color="red"))
plt.show()

This is very well but here we used [`seaborn`](https://seaborn.pydata.org/) to compute our hypothesis. Let's do that from scratch now.

## Estimating parameters

To estimate $\theta_0$ and $\theta_1$, we need 2 things:

- a cost function
- a way to update parameters given this cost function

## Predictions

Let's first define our prediction function.

In [ ]:
def predict(x: numpy.ndarray, theta_0: float, theta_1: float) -> numpy.ndarray:
  return theta_0 + x * theta_1


x_example = numpy.array([1, 2, 3])
y_example = numpy.array([4, 10, 3])
theta_0_example = 2
theta_1_example = 3

print(predict(x_example, theta_0_example, theta_1_example))

## Cost function

We can now define our cost function:

$$
  L(\theta_0, \theta_1) = \frac{1}{2n}\sum(\mathbf{x}\theta_1 + \theta_0 - \mathbf{y})^2
$$

In [ ]:
def residuals(
  x: numpy.ndarray,
  y: numpy.ndarray,
  theta_0: float,
  theta_1: float,
) -> numpy.ndarray:
  return predict(x, theta_0, theta_1) - y


print("Residuals:", residuals(x_example, y_example, theta_0_example, theta_1_example))


def cost(
  x: numpy.ndarray,
  y: numpy.ndarray,
  theta_0: float,
  theta_1: float,
) -> numpy.ndarray:
  return numpy.mean(residuals(x=x, y=y, theta_0=theta_0, theta_1=theta_1) ** 2) / 2


print("Costs:", cost(x_example, y_example, theta_0_example, theta_1_example))

## Parameters optimization

To optimize the parameters, we need to minimize the cost function:

$$\min_{\theta} \frac{1}{2n}\sum_{i=1}^n(x_i\theta_1 + \theta_0 - y_i)^2$$

To do that, we'll use gradient descent:

$$
\begin{aligned}
& \text{While not done:} \\
& \quad \theta_0 \leftarrow \theta_0 - \alpha \frac{\sum_{i=1}^n (x_i\theta_1 + \theta_0 - y_i)}{n} \\
& \quad \theta_1 \leftarrow \theta_1 - \alpha \frac{\sum_{i=1}^n (x_i\theta_1 + \theta_0 - y_i)x_i}{n} \\
\end{aligned}
$$

where $\alpha$ is the learning rate.

In [ ]:
def gradient(
  x: numpy.ndarray, y: numpy.ndarray, theta_0: float, theta_1: float
) -> tuple[float, float]:
  r = residuals(x=x, y=y, theta_1=theta_1, theta_0=theta_0)
  g_0 = r.mean()
  g_1 = x.T.dot(r) / y.size
  return g_0, g_1

In [ ]:
def gradient_descent(
  x: numpy.ndarray,
  y: numpy.ndarray,
  alpha: float = 0.1,
  nb_iter: int = 1000,
  epsilon: float = 10e-6,
) -> tuple[float, float, list[float]]:
  theta_0 = numpy.random.rand(1)[0]
  theta_1 = numpy.random.rand(1)[0]
  print(f"Initial parameters (random): θ₀ = {theta_0:.2f}, θ₁ = {theta_1:.2f}")
  initial_cost = cost(x, y, theta_0, theta_1)
  print(f"Initial cost: {initial_cost:.2f}")
  costs = [initial_cost]
  for i in range(nb_iter):
    g_0, g_1 = gradient(x, y, theta_0, theta_1)
    theta_0 -= alpha * g_0
    theta_1 -= alpha * g_1
    costs.append(cost(x, y, theta_0, theta_1))
    if costs[-2] - costs[-1] < epsilon:
      print(f"Stopping: not enough progress at iteration {i}")
      break
  print(f"Final parameters: θ₀ = {theta_0:.2f}, θ₁ = {theta_1:.2f}")
  print(f"Final cost: {costs[-1]:.2f}")
  return theta_0, theta_1, costs


theta_0, theta_1, costs = gradient_descent(x, y, 0.1, 1000)
print(f"Costs: {', '.join(map(str, costs))}")

## Hypothesis visualization

In [ ]:
inv_x = x_scaler.inverse_transform(x)
inv_y = y_scaler.inverse_transform(y)
inv_yh = y_scaler.inverse_transform(predict(x, theta_0, theta_1))

xs = numpy.arange(0, 6_000, 100)
ys = y_scaler.inverse_transform(
  predict(x_scaler.transform(xs[:, None])[:, 0], theta_0, theta_1)
)
plt.plot(inv_x, inv_y, "b.", alpha=0.10)
plt.plot(xs, ys, "r")
plt.title("True values and predictions")
plt.xlabel("Square footage")
plt.ylabel("Sale price")
plt.show()


plt.plot(inv_x, inv_y - inv_yh, "r.", alpha=0.10)
plt.title("Residuals")
plt.xlabel("Square footage")
plt.ylabel("Residuals")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Learning costs during gradient descent")
plt.xlabel("Iterations")
plt.ylabel("Error")
plt.show()

## Correction of positive asymmetry

It can be interesting to correct the asymmetry of the output variable if we notice it with a log transform such as [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html) and its inverse transform [`numpy.expm1`](https://numpy.org/doc/stable/reference/generated/numpy.expm1.html).

In [ ]:
seaborn.distplot(train_df[["SalePrice"]], fit=scipy.stats.norm)
plt.title("SalePrice distribution before normalization")
plt.show()

# Standardize y
y_scaler_log = sklearn.preprocessing.StandardScaler()
y = y_scaler_log.fit_transform(numpy.log1p(train_df[["SalePrice"]]))[:, 0]

seaborn.distplot(y, fit=scipy.stats.norm)
plt.title("SalePrice distribution after normalization")
plt.show()

In [ ]:
theta_0, theta_1, costs = gradient_descent(x, y, 0.01, 5000, 10e-6)

inv_x = x_scaler.inverse_transform(x)
inv_y = numpy.expm1(y_scaler_log.inverse_transform(y))
inv_yh = numpy.expm1(y_scaler_log.inverse_transform(predict(x, theta_0, theta_1)))

xs = numpy.arange(0, 6_000, 100)
ys = numpy.expm1(
  y_scaler_log.inverse_transform(
    predict(x_scaler.transform(xs[:, None])[:, 0], theta_0, theta_1)
  )
)
plt.plot(inv_x, inv_y, "b.", alpha=0.10)
plt.plot(xs, ys, "r")
plt.title("True values and predictions")
plt.xlabel("Square footage")
plt.ylabel("Sale price")
plt.show()

plt.plot(inv_x, inv_y - inv_yh, "r.", alpha=0.10)
plt.title("Residuals")
plt.xlabel("Square footage")
plt.ylabel("Residuals")
plt.show()

plt.plot(range(len(costs)), costs)
plt.title("Learning costs during gradient descent")
plt.xlabel("Iterations")
plt.ylabel("Error")
plt.show()

## Multivariate linear regression

Let's now see how to use several features (all of AMES data actually and not only the `GrLiveArea` column).

In [ ]:
def preprocess(train_file: str, test_file: str) -> numpy.ndarray:
  train_X = pandas.read_csv(train_file, index_col="Id")
  test_X = pandas.read_csv(test_file, index_col="Id")

  train_y = train_X.pop("SalePrice")

  all_X = pandas.concat([train_X, test_X])

  # Fill with median
  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  # Fill with mode
  cols_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  # Fill with 0
  cols_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  # Other fills
  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  # NA string fill for the rest
  all_X = all_X.fillna("NA")

  # Numerical to string for disguised categorical variables
  cols_numerical2label = ["MSSubClass"]
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  replace_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    replace_mapping[quality_column] = quality_mapping

  all_X.replace(replace_mapping, inplace=True)

  print(f"NAs number: {all_X.isnull().sum().sum()}")

  dummies = pandas.get_dummies(all_X)
  return (
    dummies.iloc[: train_X.shape[0], :].values,
    train_y.values,
    dummies.iloc[train_X.shape[0] :, :].values,
    dummies.columns,
  )

In [ ]:
X_train, y_train, X_test, columns = preprocess(
  "dataset-ames/train.csv", "dataset-ames/test.csv"
)

## Features normalization

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler().fit(X_train)
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train[:, None])[:, 0]

## Linear model training

In [ ]:
import sklearn.linear_model

linear_regression = sklearn.linear_model.LinearRegression()
linear_regression.fit(X_train_scaled, y_train_scaled)
print(linear_regression.coef_)

## Cross-validation

In [ ]:
import sklearn.compose


def score(
  model: sklearn.base.BaseEstimator,
  X: numpy.ndarray = X_train_scaled,
  y: numpy.ndarray = y_train_scaled,
) -> float:
  pipeline = sklearn.pipeline.make_pipeline(
    sklearn.preprocessing.StandardScaler(),
    sklearn.compose.TransformedTargetRegressor(
      regressor=model, transformer=sklearn.preprocessing.StandardScaler()
    ),
  )
  scores = sklearn.model_selection.cross_val_score(pipeline, X, y, cv=5, scoring="r2")
  return sum(scores) / len(scores)


score(sklearn.linear_model.LinearRegression())

## L1 and L2 regularization

In [ ]:
score_l1 = score(sklearn.linear_model.Lasso())
score_l2 = score(sklearn.linear_model.Ridge())
score_l1_l2 = score(sklearn.linear_model.ElasticNet())
print("l1, l2 and l1+l2 scores:", score_l1, score_l2, score_l1_l2)

## Hyper-parameters search

In [ ]:
model = sklearn.pipeline.make_pipeline(
  sklearn.preprocessing.StandardScaler(),
  sklearn.compose.TransformedTargetRegressor(
    regressor=sklearn.linear_model.ElasticNet(),
    transformer=sklearn.preprocessing.StandardScaler(),
  ),
)
params = dict(transformedtargetregressor__regressor__alpha=[0.1, 1])
rscv = sklearn.model_selection.RandomizedSearchCV(model, params, scoring="r2")
search = rscv.fit(X_train_scaled, y_train_scaled)
print(f"RSCV parameters: {search.best_params_}")


# With Nevergrad
def loss(alpha: float, l1_ratio: float) -> float:
  return -score(sklearn.linear_model.ElasticNet(alpha=alpha, l1_ratio=l1_ratio))


parametrization = nevergrad.p.Instrumentation(
  alpha=nevergrad.p.Scalar(lower=0.0, upper=3.0),
  l1_ratio=nevergrad.p.Scalar(lower=0.0, upper=1.0),
)

optimizer = nevergrad.optimizers.NGOpt(parametrization=parametrization, budget=50)
recommendation = optimizer.minimize(loss)
print(f"Nevergrad parameters: {recommendation.kwargs}")

In [ ]:
print(recommendation.kwargs)
best_model = sklearn.linear_model.ElasticNet(**recommendation.kwargs)
best_model.fit(X_train_scaled, y_train_scaled)
score(best_model)

## Feature importance

In [ ]:
feature_importances = sklearn.inspection.permutation_importance(
  best_model, X_train_scaled, y_train_scaled, n_repeats=50
)

In [ ]:
series = pandas.Series(feature_importances.importances_mean, index=columns)
series = series.sort_values(ascending=False).iloc[:10]
series.plot.bar()

## Polynomial features

In [ ]:
top10 = (-feature_importances.importances_mean).argsort()[:10]


def poly_features(array: numpy.ndarray) -> numpy.ndarray:
  polynomial_features = sklearn.preprocessing.PolynomialFeatures(
    2, interaction_only=True
  )
  return polynomial_features.fit_transform(array[:, top10])


X_train_poly = poly_features(X_train_scaled)
X_test_poly = poly_features(X_test_scaled)

score(sklearn.linear_model.ElasticNet(), X_train_poly)